## Example 1: Safe conversions that return the input on failure

In [ ]:
import fastnumbers
from fastnumbers import try_float, try_int, try_real

print("fastnumbers", fastnumbers.__version__, "from", fastnumbers.__file__)
assert try_float("3.25") == 3.25 and try_float("  -1e3 ") == -1000.0
assert try_int("42") == 42 and try_int("4.2") == "4.2"
assert try_real("1e5") == 100000 and isinstance(try_real("1e5"), int)
assert try_real("2.5") == 2.5
assert try_float("not a number") == "not a number"
assert try_int("x", on_fail=None) is None and try_int("x", on_fail=len) == 1
print([try_real(s) for s in ("12", "12.0", "1.5e2", "abc")])

## Example 2: The same answers as Python's own float() and int()

In [ ]:
import math
from fastnumbers import try_float, try_int

samples = ["0", "-0.0", "1e-308", "1.7976931348623157e308", "123456789012345678901234567890",
           "0.1", "  42  ", "inf", "-infinity", "nan", "12345678.87654321"]
for text in samples:
    ours = try_float(text)
    theirs = float(text)
    assert (math.isnan(ours) and math.isnan(theirs)) or ours == theirs, (text, ours, theirs)
assert try_int("1" * 40) == int("1" * 40)  # long digit strings take the eight-digit fast path
assert [try_int(s) for s in ("12345678", "123456789", "00000000")] == [12345678, 123456789, 0]
assert try_float("1_000") == "1_000" and try_float("1_000", allow_underscores=True) == 1000.0  # opt-in, unlike float()
print("all", len(samples), "strings parse as Python parses them")

## Example 3: Checking without converting

In [ ]:
import fastnumbers
from fastnumbers import check_float, check_int, check_real, check_intlike

assert check_real("1.5") and check_real("10") and not check_real("ten")
assert check_int("10") and not check_int("10.0")
assert check_float("10.0") and check_float("1e3")
assert not check_float("1e3", consider=fastnumbers.NUMBER_ONLY) and check_float(1.5, consider=fastnumbers.NUMBER_ONLY)
assert check_intlike("10.0") and not check_intlike("10.5")
assert not check_real("0x1A")  # hexadecimal is not a decimal number
print("checks behave as documented")

## Example 4: Unicode digits and numbers in other scripts

In [ ]:
from fastnumbers import try_float, try_int

assert try_float("１２３") == 123.0  # full-width digits
assert try_int("٣") == 3  # an Arabic-Indic digit
assert try_float("½") == 0.5  # a single numeric character
assert try_int("½") == "½"  # not an integer
print(try_float("١٢٣٫٥"), "(an Arabic-Indic number with its own decimal mark stays text)")

## Example 5: natsort uses it as its number parser

In [ ]:
import natsort
from natsort.compat import fastnumbers as compat

print("natsort", natsort.__version__, "parses numbers with", compat.try_float)
assert compat.try_float.__module__ == "fastnumbers.fastnumbers", compat.try_float.__module__
items = ["a10", "a2", "a1.5", "a-3", "a1e1"]
ordered = natsort.natsorted(items, alg=natsort.ns.FLOAT | natsort.ns.SIGNED)
print(ordered)
assert ordered == ["a-3", "a1.5", "a2", "a10", "a1e1"], ordered
assert natsort.natsorted(["v1.10", "v1.9", "v1.2"]) == ["v1.2", "v1.9", "v1.10"]